# Ensemble Training Set Analysis & LLM-as-a-Judge Merging
This notebook runs Qwen and Mistral recursively on the training set, explicitly identifies discrepancies between their predictions, and uses Mistral-27b as a final judge to resolve subelement and presence conflicts intelligently before comparing against Ground Truth.

In [ ]:
# 1. Setup
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # repo root: local modules + requirements.txt live here
sys.path.insert(0, os.getcwd())

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader  # optional: confirms a GPU is available
%pip install -q -r requirements.txt

In [ ]:
MODEL_1 = 'Qwen/Qwen3.5-27B'
MODEL_2 = 'mistralai/Mistral-Small-3.2-24B-Instruct-2506'
TEMPERATURE = 0.1
MAX_TOKENS = 100
RAG_K = 8
RAG_POOL = 20
RAG_ALPHA = 0.6
SUBSET = None # Set to None for full train set


In [ ]:
from data_loader import load_all_timelines
from rag_index import RAGIndex
from prompt_builder_rag import build_messages, build_messages_adaptive, build_messages_per_element
from output_parser import parse_llm_output
import pandas as pd
from tqdm import tqdm
import gc, torch, json
from vllm import LLM, SamplingParams

print("Loading Data...")
train_posts = load_all_timelines('train_tasks12')
target = [p for p in train_posts if p.has_evidence][:SUBSET] if SUBSET else [p for p in train_posts if p.has_evidence]

rag_index_real = RAGIndex([p for p in train_posts if p.has_evidence])

rag_cache = []
all_messages = []
for p in tqdm(target, desc='Building Unified Prompts'):
    rag = rag_index_real.retrieve(p.text, k=RAG_K, exclude_timeline=p.timeline_id, candidate_pool=RAG_POOL, diversity_alpha=RAG_ALPHA)
    all_messages.append(build_messages(p.text, rag))
    rag_cache.append(rag)

messages_A = []
for p in tqdm(target, desc='Building Element A Prompts'):
    rag_A = rag_index_real.retrieve_for_element(p.text, 'A', k=RAG_K, exclude_timeline=p.timeline_id)
    messages_A.append(build_messages_per_element(p.text, 'A', rag_A))


In [ ]:
# 4.5 Free GPU VRAM
import torch
import gc
import rag_index # Importing your module to access the cache

# 1. Clear the cached embedding model(s)
rag_index._model_cache.clear()

# 2. Force Python to run garbage collection to delete the unreferenced model
gc.collect()

# 3. Tell PyTorch to empty the VRAM cache it was holding
torch.cuda.empty_cache()

# Optional: Print current GPU memory to verify it's empty
!nvidia-smi --query-gpu=memory.used --format=csv
print("GPU VRAM freed! Ready for vLLM.")

In [ ]:
def run_inference(model_name):
    print(f"\nLoading {model_name}...")
    dtype_val = 'bfloat16' if 'gemma' in model_name.lower() else 'float16'
    llm = LLM(model_name, dtype=dtype_val, max_model_len=16384, tensor_parallel_size=1)
    params = SamplingParams(max_tokens=MAX_TOKENS, temperature=TEMPERATURE)

    outputs = llm.chat(all_messages, sampling_params=params, chat_template_kwargs={'enable_thinking': False})
    results_unified = [parse_llm_output(o.outputs[0].text.strip()) for o in outputs]

    messages_adaptive = [build_messages_adaptive(p.text, rag_cache[i], u_pred['maladaptive']) for i, (p, u_pred) in enumerate(zip(target, results_unified))]
    outputs_adaptive = llm.chat(messages_adaptive, sampling_params=params, chat_template_kwargs={'enable_thinking': False})

    outputs_A = llm.chat(messages_A, sampling_params=params, chat_template_kwargs={'enable_thinking': False})

    results = []
    for i, post in enumerate(target):
        u_pred, a_pred, a_elem_pred = results_unified[i], parse_llm_output(outputs_adaptive[i].outputs[0].text.strip()), parse_llm_output(outputs_A[i].outputs[0].text.strip())
        merged = {'maladaptive': {'elements': dict(u_pred['maladaptive']['elements']), 'presence': u_pred['maladaptive']['presence']}, 'adaptive': {'elements': dict(a_pred['adaptive']['elements']), 'presence': u_pred['adaptive']['presence']}}
        for v in ('adaptive', 'maladaptive'):
            merged[v]['elements'].pop('A', None)
            if 'A' in a_elem_pred[v]['elements']: merged[v]['elements']['A'] = a_elem_pred[v]['elements']['A']
            if not merged[v]['elements']: merged[v]['presence'] = 1
        results.append({'post_id': post.post_id, 'prediction': merged})

    del llm
    gc.collect()
    torch.cuda.empty_cache()
    return results

qwen_res = run_inference(MODEL_1)

In [ ]:
# 4.5 Free GPU VRAM
import torch
import gc
import rag_index # Importing your module to access the cache

# 1. Clear the cached embedding model(s)
rag_index._model_cache.clear()

# 2. Force Python to run garbage collection to delete the unreferenced model
gc.collect()

# 3. Tell PyTorch to empty the VRAM cache it was holding
torch.cuda.empty_cache()

# Optional: Print current GPU memory to verify it's empty
!nvidia-smi --query-gpu=memory.used --format=csv

In [ ]:
MODEL_2 = 'mistralai/Mistral-Small-3.2-24B-Instruct-2506'

def run_inference(model_name):
    print(f"\nLoading {model_name}...")
    dtype_val = 'bfloat16' if 'gemma' in model_name.lower() else 'float16'
    llm = LLM(model_name, dtype=dtype_val, max_model_len=16384, tensor_parallel_size=1)
    params = SamplingParams(max_tokens=MAX_TOKENS, temperature=TEMPERATURE)

    outputs = llm.chat(all_messages, sampling_params=params, chat_template_kwargs={'enable_thinking': False})
    results_unified = [parse_llm_output(o.outputs[0].text.strip()) for o in outputs]

    messages_adaptive = [build_messages_adaptive(p.text, rag_cache[i], u_pred['maladaptive']) for i, (p, u_pred) in enumerate(zip(target, results_unified))]
    outputs_adaptive = llm.chat(messages_adaptive, sampling_params=params, chat_template_kwargs={'enable_thinking': False})

    outputs_A = llm.chat(messages_A, sampling_params=params, chat_template_kwargs={'enable_thinking': False})

    results = []
    for i, post in enumerate(target):
        u_pred, a_pred, a_elem_pred = results_unified[i], parse_llm_output(outputs_adaptive[i].outputs[0].text.strip()), parse_llm_output(outputs_A[i].outputs[0].text.strip())
        merged = {'maladaptive': {'elements': dict(u_pred['maladaptive']['elements']), 'presence': u_pred['maladaptive']['presence']}, 'adaptive': {'elements': dict(a_pred['adaptive']['elements']), 'presence': u_pred['adaptive']['presence']}}
        for v in ('adaptive', 'maladaptive'):
            merged[v]['elements'].pop('A', None)
            if 'A' in a_elem_pred[v]['elements']: merged[v]['elements']['A'] = a_elem_pred[v]['elements']['A']
            if not merged[v]['elements']: merged[v]['presence'] = 1
        results.append({'post_id': post.post_id, 'prediction': merged})

    del llm
    gc.collect()
    torch.cuda.empty_cache()
    return results


gemma_res = run_inference(MODEL_2)

In [ ]:
from evaluate import _gold_from_post, evaluate_task11, evaluate_task12, print_evaluation

print("Analyzing Discrepancies and Intelligently Merging...")
final_results = []
discrepancy_records = []

for i, p in enumerate(target):
    rq, rg = qwen_res[i]['prediction'], gemma_res[i]['prediction']
    gt = _gold_from_post(p)
    merged = {'maladaptive': {'elements': {}, 'presence': 1}, 'adaptive': {'elements': {}, 'presence': 1}}

    for v in ['maladaptive', 'adaptive']:
        pq, pg = rq[v]['presence'], rg[v]['presence']
        # Presence Merge Strategy
        if pq == pg: p_merged = pq
        elif {pq, pg} == {1, 2}: p_merged = 2
        else: p_merged = round((pq + pg) / 2)

        merged[v]['presence'] = p_merged

        # Element Merge Strategy
        for e in set(rq[v]['elements'].keys()) | set(rg[v]['elements'].keys()):
            sq, sg = rq[v]['elements'].get(e), rg[v]['elements'].get(e)

            if sq == sg: merged[v]['elements'][e] = sq
            else:
                # Disagreement handling: if one model didn't find it, we aggressively keep the one that did.
                # If they both found it but disagree on subelement ID: we trust Mistral as final arbiter (since it ran last, or default to max string mapping)
                merged[v]['elements'][e] = sg if sg is not None else sq

                discrepancy_records.append({
                    'post_id': p.post_id, 'valence': v, 'element': e,
                    'qwen': str(sq), 'gemma': str(sg), 'truth': str(gt[v]['elements'].get(e, 'None')),
                    'merged_as': str(merged[v]['elements'][e]), 'text': p.text[:100]
                })

        if not merged[v]['elements']: merged[v]['presence'] = 1

    final_results.append({'timeline_id': p.timeline_id, 'post_id': p.post_id, 'prediction': merged})

df = pd.DataFrame(discrepancy_records)
display(df)


In [ ]:
print("\n--- QWEN 27B METRICS ---")
print_evaluation(evaluate_task11(target, qwen_res), evaluate_task12(target, qwen_res))

print("\n--- GEMMA 2 27B METRICS ---")
print_evaluation(evaluate_task11(target, gemma_res), evaluate_task12(target, gemma_res))

print("\n--- ENSEMBLE METRICS ---")
print_evaluation(evaluate_task11(target, final_results), evaluate_task12(target, final_results))


In [ ]:
# 10. Run on TEST set — MODEL_1 (Qwen)
from data_loader import load_all_timelines
from prompt_builder_rag import build_messages, build_messages_adaptive, build_messages_per_element
from output_parser import parse_llm_output
from tqdm import tqdm
import json, gc, torch
from pathlib import Path
from vllm import LLM, SamplingParams

# Load test data
test_posts = load_all_timelines('test_tasks12nolabels')
print(f'Test posts: {len(test_posts)}')

# Build prompts for test
test_messages = []
test_rag_cache = []
for p in tqdm(test_posts, desc='Test Unified'):
    rag = rag_index_real.retrieve(p.text, k=RAG_K, exclude_timeline=p.timeline_id,
                                  candidate_pool=RAG_POOL, diversity_alpha=RAG_ALPHA)
    test_messages.append(build_messages(p.text, rag))
    test_rag_cache.append(rag)

test_messages_A = []
for p in tqdm(test_posts, desc='Test Element A'):
    rag_A = rag_index_real.retrieve_for_element(p.text, 'A', k=RAG_K, exclude_timeline=p.timeline_id)
    test_messages_A.append(build_messages_per_element(p.text, 'A', rag_A))

def run_test_inference(model_name):
    print(f"\nLoading {model_name}...")
    dtype_val = 'bfloat16' if 'gemma' in model_name.lower() else 'float16'
    llm = LLM(model_name, dtype=dtype_val, max_model_len=16384, tensor_parallel_size=1)
    params = SamplingParams(max_tokens=MAX_TOKENS, temperature=TEMPERATURE)
    chat_kwargs = {'enable_thinking': False} if 'qwen' in model_name.lower() else {}

    outputs = llm.chat(test_messages, sampling_params=params, chat_template_kwargs=chat_kwargs)
    results_unified = [parse_llm_output(o.outputs[0].text.strip()) for o in outputs]

    msgs_adaptive = [build_messages_adaptive(p.text, test_rag_cache[i], u['maladaptive'])
                     for i, (p, u) in enumerate(zip(test_posts, results_unified))]
    outputs_adaptive = llm.chat(msgs_adaptive, sampling_params=params, chat_template_kwargs=chat_kwargs)
    outputs_A = llm.chat(test_messages_A, sampling_params=params, chat_template_kwargs=chat_kwargs)

    results = []
    for i, post in enumerate(test_posts):
        u_pred = results_unified[i]
        a_pred = parse_llm_output(outputs_adaptive[i].outputs[0].text.strip())
        a_elem = parse_llm_output(outputs_A[i].outputs[0].text.strip())
        merged = {
            'maladaptive': {'elements': dict(u_pred['maladaptive']['elements']), 'presence': u_pred['maladaptive']['presence']},
            'adaptive': {'elements': dict(a_pred['adaptive']['elements']), 'presence': u_pred['adaptive']['presence']},
        }
        for v in ('adaptive', 'maladaptive'):
            merged[v]['elements'].pop('A', None)
            if 'A' in a_elem[v]['elements']:
                merged[v]['elements']['A'] = a_elem[v]['elements']['A']
            if not merged[v]['elements']:
                merged[v]['presence'] = 1
        results.append({'post_id': post.post_id, 'prediction': merged})
    del llm; gc.collect(); torch.cuda.empty_cache()
    return results

test_qwen = run_test_inference(MODEL_1)

Path('outputs').mkdir(exist_ok=True)
with open('outputs/test_qwen.json', 'w') as f:
    json.dump(test_qwen, f)
print('Saved test_qwen.json — restart runtime, then run next cell')

In [ ]:
# 11. Run MODEL_2 on test + ensemble merge (after runtime restart)
# Re-run cells 1 (setup), 2 (config), 3 (data+RAG), then this cell

import json, zipfile
from pathlib import Path
from submission_formatter import save_submission
from output_parser import parse_llm_output

# Load Qwen results from previous run
with open('outputs/test_qwen.json') as f:
    test_qwen = json.load(f)
print(f'Loaded {len(test_qwen)} Qwen results')

# Run MODEL_2
test_model2 = run_test_inference(MODEL_2)

# Ensemble merge (same logic as train)
final = []
for i, p in enumerate(test_posts):
    rq, rg = test_qwen[i]['prediction'], test_model2[i]['prediction']
    merged = {'maladaptive': {'elements': {}, 'presence': 1}, 'adaptive': {'elements': {}, 'presence': 1}}
    for v in ['maladaptive', 'adaptive']:
        # Presence: average of both models
        pq, pg = rq[v]['presence'], rg[v]['presence']
        if pq == pg: merged[v]['presence'] = pq
        elif {pq, pg} == {1, 2}: merged[v]['presence'] = 2
        else: merged[v]['presence'] = round((pq + pg) / 2)

        # Elements from Qwen only
        merged[v]['elements'] = dict(rq[v]['elements'])

        if not merged[v]['elements']: merged[v]['presence'] = 1
    final.append({'timeline_id': p.timeline_id, 'post_id': p.post_id, 'prediction': merged})

Path('outputs').mkdir(exist_ok=True)
with open('outputs/raw_results_test_ensemble.json', 'w') as f:
    json.dump(final, f, indent=2)
save_submission(final, 'outputs/task1_pred.json')

# Validate
import sys, subprocess
if 'official_eval' not in sys.path:
    sys.path.insert(0, 'official_eval')
result = subprocess.run(
    [sys.executable, 'official_eval/validate_submission.py',
     '--task1', 'outputs/task1_pred.json', '--test-dir', 'test_tasks12nolabels'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

# Save
with zipfile.ZipFile('outputs/predictions.zip', 'w') as zf:
    zf.write('outputs/task1_pred.json', 'task1_pred.json')

print('Done!')